# Chakravyuh — env exploration (no GPU, no LoRA, no training)

5-minute tutorial. Reset the env, step through one episode, score with the rule-based scripted analyzer, see the full reward decomposition. Runs on a laptop CPU.

**What this notebook does NOT do:**

- It does **not** load the LoRA adapter (that's GPU-bound; see `notebooks/v2_retrain_safe.ipynb` for the trained-model path).
- It does **not** run GRPO training (`training/grpo_analyzer.py`).
- It does **not** require any HF Hub / Colab / OpenAI / Anthropic API credits.

Goal: give you the env contract in your hands so you understand what `reset` / `step` / `state` actually return.

## Setup

Before running this notebook:

```bash
git clone https://github.com/UjjwalPardeshi/Chakravyuh && cd Chakravyuh
python -m venv .venv && source .venv/bin/activate
pip install -e .
jupyter notebook notebooks/env_exploration.ipynb
```

The `pip install -e .` step makes the `chakravyuh_env` package importable in the kernel.

In [ ]:
from chakravyuh_env import ChakravyuhOpenEnv, ChakravyuhAction
from chakravyuh_env.agents.analyzer import ScriptedAnalyzer
from chakravyuh_env.schemas import ChatMessage, Observation

env = ChakravyuhOpenEnv()
obs = env.reset(seed=42)

print(f"Scenario: scam_category={obs.scam_category}, victim_profile={obs.victim_profile}")
print(f"Turn: {obs.turn} (decision_index={obs.decision_index})")
print(f"Schema version: {obs.schema_version}")
print(f"\nFirst chat message:")
for msg in obs.chat_history:
    print(f"  T{msg['turn']} {msg['sender']:>10}: {msg['text']}")

## Step 2 — score with the rule-based analyzer (CPU-only)

The scripted analyzer is keyword-driven; it's the v0 baseline (we replaced it with a Qwen2.5-7B LoRA in v2). Useful here because it runs in milliseconds and doesn't need any model weights.

In [ ]:
analyzer = ScriptedAnalyzer(seed=42)
scammer_text = next(m['text'] for m in obs.chat_history if m['sender'] == 'scammer')
scoring_obs = Observation(
    agent_role='analyzer',
    turn=1,
    chat_history=[ChatMessage(sender='scammer', turn=1, text=scammer_text)],
)
result = analyzer.act(scoring_obs)
print(f"Score: {result.score:.2f}")
print(f"Signals: {[s.value if hasattr(s, 'value') else s for s in (result.signals or [])]}")
print(f"Explanation: {result.explanation}")

## Step 3 — step the env, see the reward decomposition

Submit the analyzer's verdict back to the env via `step()`. The env runs the rest of the episode (Bank Monitor decision, victim outcome) and returns the terminal observation with the full reward breakdown.

In [ ]:
action = ChakravyuhAction(
    score=float(result.score),
    signals=[s.value if hasattr(s, 'value') else s for s in (result.signals or [])],
    explanation=result.explanation or '',
)
next_obs = env.step(action)
while not next_obs.done:
    next_obs = env.step(action)

print(f"Episode terminated at turn {next_obs.turn}")
print(f"Reward: {next_obs.reward:.3f}")
print(f"\nOutcome:")
for k, v in (next_obs.outcome or {}).items():
    print(f"  {k:<22}: {v}")
print(f"\nReward breakdown (per-rubric leaf):")
for k, v in (next_obs.reward_breakdown or {}).items():
    if isinstance(v, (int, float)):
        print(f"  {k:<22}: {v:+.3f}")

## Step 4 — reproduce a 100-episode rollout

Useful sanity check: same seed, same trajectory.

In [ ]:
import collections
outcomes = collections.Counter()
for seed in range(100):
    e = ChakravyuhOpenEnv()
    o = e.reset(seed=seed)
    text = next(m['text'] for m in o.chat_history if m['sender'] == 'scammer')
    sa = ScriptedAnalyzer(seed=seed)
    sa_res = sa.act(Observation(
        agent_role='analyzer', turn=1,
        chat_history=[ChatMessage(sender='scammer', turn=1, text=text)],
    ))
    a = ChakravyuhAction(score=float(sa_res.score), signals=[], explanation='')
    nxt = e.step(a)
    while not nxt.done:
        nxt = e.step(a)
    flagged = (nxt.outcome or {}).get('analyzer_flagged', False)
    outcomes['flagged' if flagged else 'missed'] += 1

print(f"Across 100 seeded episodes with the rule-based analyzer:")
print(f"  flagged: {outcomes['flagged']}")
print(f"  missed:  {outcomes['missed']}")
print(f"\nThis is the scripted baseline. The v2 LoRA closes the gap to {97}% on the novel split — see logs/eval_v2.json.")

## Where to next

- **Run the live red-team tab in the demo:** `python -m server.demo_ui` then visit `http://localhost:7860` — paste any scam, see the v1 reward profile vs v2 reward profile decomposition side-by-side.
- **Read the reward decomposition:** [`chakravyuh_env/rubrics.py`](../chakravyuh_env/rubrics.py) — `AnalyzerRubricV2` has 8 children. The v1→v2 weight changes are documented in `V2_WEIGHTS` and tested in `tests/test_v2_reward_parity.py`.
- **Reproduce the bench numbers:** `make reproduce` (~10 min CPU with cached scores; needs GPU for fresh inference).
- **Train your own analyzer:** [`training/grpo_analyzer.py`](../training/grpo_analyzer.py) (GPU required).